In [21]:
#!uv pip install scikit-learn
#!uv pip install imbalanced-learn
#!uv pip install seaborn

# I wasted more time trying to understand your cviko 6 project rather than implementing my own, so I stopped unproductive time loss and made my own


In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import sys
import time

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score
)

# ----------------------------
# CONFIG
# ----------------------------
N_REPLICATIONS = 15
OUTPUT_CSV = "model_metrics_over_replicas.csv"


# ----------------------------
# DATA
# ----------------------------
data = load_breast_cancer()
X, y = data.data, data.target


# ----------------------------
# MODELS
# ----------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=10000, solver="liblinear"),
    "Random Forest": RandomForestClassifier(random_state=42)
}

param_grids = {
    "Logistic Regression": {
        "C": [0.1, 1, 10]
    },
    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, None],
        "min_samples_split": [2, 5]
    }
}


# ----------------------------
# STORAGE
# ----------------------------
results = []


def print_progress(current, total, model_name):
    percent = (current / total) * 100
    bar_length = 30
    filled = int(bar_length * current // total)
    bar = "█" * filled + "-" * (bar_length - filled)

    sys.stdout.write(
        f"\r{model_name} | {bar} | {current}/{total} ({percent:.1f}%)"
    )
    sys.stdout.flush()


# ----------------------------
# EXPERIMENT LOOP
# ----------------------------
for model_name, model in models.items():

    grid = GridSearchCV(
        model,
        param_grids[model_name],
        cv=5,
        scoring="accuracy"
    )

    for rep in range(N_REPLICATIONS):

        print_progress(rep + 1, N_REPLICATIONS, model_name)

        grid.fit(X, y)
        best_model = grid.best_estimator_

        y_pred = best_model.predict(X)

        try:
            y_proba = best_model.predict_proba(X)[:, 1]
            roc = roc_auc_score(y, y_proba)
        except:
            roc = np.nan

        results.append({
            "model": model_name,
            "replication": rep + 1,
            "accuracy": accuracy_score(y, y_pred),
            "f1": f1_score(y, y_pred),
            "recall": recall_score(y, y_pred),
            "precision": precision_score(y, y_pred),
            "roc_auc": roc,
            "best_params": str(grid.best_params_)
        })

    print("\n")


# ----------------------------
# SAVE CSV (IMPORTANT FIX)
# ----------------------------
df = pd.DataFrame(results)
df.to_csv(OUTPUT_CSV, index=False)

print("\nSaved metrics per replication to:", OUTPUT_CSV)


# ----------------------------
# QUICK INSIGHT (OPTIONAL BUT USEFUL)
# ----------------------------
summary = df.groupby("model")[["accuracy", "f1", "recall", "precision", "roc_auc"]].mean()
print("\nMEAN PERFORMANCE:")
print(summary)

Logistic Regression | ██████████████████████████████ | 15/15 (100.0%)

Random Forest | ██████████████████████████████ | 15/15 (100.0%)


Saved metrics per replication to: model_metrics_over_replicas.csv

MEAN PERFORMANCE:
                     accuracy        f1   recall  precision   roc_auc
model                                                                
Logistic Regression  0.959578  0.968011  0.97479   0.961326  0.994768
Random Forest        0.996485  0.997207  1.00000   0.994429  1.000000


In [3]:
import os
import matplotlib.pyplot as plt

PLOT_DIR = "plots"
os.makedirs(PLOT_DIR, exist_ok=True)

metrics = ["accuracy", "f1", "recall", "precision", "roc_auc"]

df = pd.DataFrame(results)

def save_plot(name):
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_DIR, name))
    plt.close()


# ----------------------------
# 1. BOX PLOTS (BEST FOR COMPARISON)
# ----------------------------
for metric in metrics:
    plt.figure()

    data_to_plot = [
        df[df["model"] == model][metric].values
        for model in df["model"].unique()
    ]

    plt.boxplot(data_to_plot, labels=df["model"].unique())
    plt.title(f"Comparison of {metric} (Boxplot)")
    plt.ylabel(metric)

    save_plot(f"boxplot_{metric}.png")


# ----------------------------
# 2. MEAN BAR CHART (CLEAR WINNER VIEW)
# ----------------------------
mean_scores = df.groupby("model")[metrics].mean()

mean_scores.plot(kind="bar")
plt.title("Average Metrics Comparison")
plt.ylabel("Score")

save_plot("mean_metrics_comparison.png")


# ----------------------------
# 3. LINE PLOT OVER REPLICATIONS (STABILITY VIEW)
# ----------------------------
for metric in metrics:
    plt.figure()

    for model in df["model"].unique():
        subset = df[df["model"] == model].sort_values("replication")

        plt.plot(
            subset["replication"],
            subset[metric],
            marker="o",
            label=model
        )

    plt.title(f"{metric} over Replications")
    plt.xlabel("Replication")
    plt.ylabel(metric)
    plt.legend()

    save_plot(f"trend_{metric}.png")


print(f"\nAll plots saved to: {PLOT_DIR}/")

/tmp/ipykernel_47875/1732136684.py:28: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_to_plot, labels=df["model"].unique())
/tmp/ipykernel_47875/1732136684.py:28: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_to_plot, labels=df["model"].unique())
/tmp/ipykernel_47875/1732136684.py:28: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  plt.boxplot(data_to_plot, labels=df["model"].unique())
/tmp/ipykernel_47875/1732136684.py:28: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.1


All plots saved to: plots/
